In [1]:
!pip install opencv-python-headless==4.5.3.56

     |████████████████████████████████| 37.1 MB 2.8 MB/s eta 0:00:01
You should consider upgrading via the '/usr/bin/python -m pip install --upgrade pip' command.


In [2]:
!pip install pandas==1.4.4
!pip install numpy==1.19.4
!pip install tqdm
!pip install scikit-learn==0.24.2
!pip install Pillow==8.0.1
!pip install matplotlib==3.3.4
!pip install tensorflow==2.4.0
!pip install kneed
!pip install numpy==1.19.5

     |████████████████████████████████| 11.7 MB 2.2 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 50.4 MB/s eta 0:00:01
You should consider upgrading via the '/usr/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/usr/bin/python -m pip install --upgrade pip' command.
You should consider upgrading via the '/usr/bin/python -m pip install --upgrade pip' command.
     |████████████████████████████████| 24.9 MB 2.9 MB/s eta 0:00:01
     |████████████████████████████████| 301 kB 54.8 MB/s eta 0:00:01
You should consider upgrading via the '/usr/bin/python -m pip install --upgrade pip' command.
     |████████████████████████████████| 2.2 MB 2.2 MB/s eta 0:00:01
You should consider upgrading via the '/usr/bin/python -m pip install --upgrade pip' command.
     |████████████████████████████████| 11.6 MB 2.8 MB/s eta 0:00:01
     |████████████████████████████████| 1.2 MB 75.3 MB/s eta 0:00:01
You should consider upgrading via the '/usr/bin/

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tqdm
import tensorflow as tf
from PIL import Image
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
data=pd.read_csv("data.csv")
labels=[]
links=[]
for idx,row in tqdm.tqdm(data.iterrows()):
    labels.append(row['labels'])
    links.append(row['path'])

13320it [00:00, 14947.77it/s]


In [4]:
import cv2
import numpy as np
from scipy.stats import entropy

from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import img_to_array

###############################################################
# Load pretrained ResNet50
###############################################################

base_model = ResNet50(
    weights='imagenet',
    include_top=False,
    pooling='avg'
)

###############################################################
# Feature Extraction
###############################################################

def extract_feature(frame):

    image = cv2.resize(frame, (224,224))
    image = img_to_array(image)
    image = np.expand_dims(image, axis=0)
    image = preprocess_input(image)

    feature = base_model.predict(image, verbose=0)[0]

    return feature

###############################################################
# Shannon Entropy
###############################################################

def frame_entropy(frame):

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    hist = cv2.calcHist(
        [gray],
        [0],
        None,
        [256],
        [0,256]
    )

    hist = hist.ravel()

    hist /= hist.sum()

    return entropy(hist, base=2)

###############################################################
# Main Function
###############################################################

def extract_keyframes(video_path,
                      z_threshold=1.0):

    cap = cv2.VideoCapture(video_path)

    frames = []

    entropies = []

    features = []

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        frames.append(frame)

        entropies.append(
            frame_entropy(frame)
        )

        features.append(
            extract_feature(frame)
        )

    cap.release()

    entropies = np.array(entropies)

    ###########################################################
    # Feature Difference
    ###########################################################

    feature_diff = [0]

    for i in range(1, len(features)):

        d = np.linalg.norm(
            features[i]-features[i-1]
        )

        feature_diff.append(d)

    feature_diff = np.array(feature_diff)

    ###########################################################
    # Normalize
    ###########################################################

    H = (entropies-entropies.min()) / \
        (entropies.max()-entropies.min()+1e-8)

    D = (feature_diff-feature_diff.min()) / \
        (feature_diff.max()-feature_diff.min()+1e-8)

    ###########################################################
    # Adaptive alpha beta
    ###########################################################

    sigma_H = np.std(H)

    sigma_D = np.std(D)

    alpha = sigma_H / (sigma_H + sigma_D + 1e-8)

    beta = sigma_D / (sigma_H + sigma_D + 1e-8)

    ###########################################################
    # Frame Importance Score
    ###########################################################

    score = alpha*H + beta*D

    ###########################################################
    # Z-score
    ###########################################################

    mean = np.mean(score)

    std = np.std(score)

    z = (score-mean)/(std+1e-8)

    ###########################################################
    # Keyframes
    ###########################################################

    keyframes = []

    keyframe_indices = []

    for i in range(len(frames)):

        if z[i] > z_threshold:

            keyframes.append(frames[i])

            keyframe_indices.append(i)

    return keyframes,\
           keyframe_indices,\
           score,\
           z,\
           alpha,\
           beta

94773248/94765736 [==============================] - 7s 0us/step


In [11]:
keyframes, indices, score, z, alpha, beta = extract_keyframes(
    links[0],
    z_threshold=1.0
)

print("Alpha :", alpha)
print("Beta  :", beta)
print("Number of keyframes :", len(indices))
print(indices)

Alpha : 0.5434501865631841
Beta  : 0.45654979566657583
Number of keyframes : 28
[75, 77, 79, 83, 85, 87, 91, 93, 95, 98, 100, 102, 103, 104, 106, 107, 108, 109, 110, 112, 114, 116, 118, 120, 123, 125, 133, 143]


In [10]:
links[0]

'/workspace/data/data/UCF-101/ApplyEyeMakeup/v_ApplyEyeMakeup_g01_c01.avi'

In [ ]:
for link in links:
    keyframes, indices, score, z, alpha, beta = extract_keyframes(
    link,
    z_threshold=1.0
 )
    print("Alpha :", alpha)
    print("Beta  :", beta)
    print("Number of keyframes :", len(indices))
    print(indices)
    

        

Alpha : 0.5434501865631841
Beta  : 0.45654979566657583
Number of keyframes : 28
[75, 77, 79, 83, 85, 87, 91, 93, 95, 98, 100, 102, 103, 104, 106, 107, 108, 109, 110, 112, 114, 116, 118, 120, 123, 125, 133, 143]
Alpha : 0.53213145750176
Beta  : 0.46786852078935626
Number of keyframes : 19
[2, 6, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 34, 46, 79, 121, 122]
Alpha : 0.4280205307882604
Beta  : 0.5719794485496833
Number of keyframes : 51
[2, 4, 6, 8, 52, 60, 81, 102, 110, 112, 114, 121, 123, 131, 133, 135, 144, 146, 148, 152, 158, 160, 162, 164, 169, 171, 175, 179, 181, 185, 194, 196, 200, 202, 212, 214, 219, 221, 223, 225, 227, 231, 233, 235, 237, 239, 241, 244, 246, 252, 256]
Alpha : 0.5664252927572098
Beta  : 0.4335746850482912
Number of keyframes : 45
[137, 139, 141, 142, 143, 144, 145, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 160, 162, 164, 166, 168, 170, 172, 174, 176, 180, 182, 185, 187, 189, 191, 193, 195, 197, 201, 203, 207, 210, 212, 213, 214, 215, 216]
Alpha 

In [5]:
import os
import cv2
from concurrent.futures import ProcessPoolExecutor, as_completed

output_dir = "keyframes"
os.makedirs(output_dir, exist_ok=True)


def process_video(args):
    video_num, link = args

    # Extract keyframes
    keyframes, indices, score, z, alpha, beta = extract_keyframes(
        link,
        z_threshold=1.0
    )

    # Create output folder
    video_folder = os.path.join(output_dir, str(video_num))
    os.makedirs(video_folder, exist_ok=True)

    # Save keyframes
    for i, frame in enumerate(keyframes, start=1):
        cv2.imwrite(os.path.join(video_folder, f"{i}.jpg"), frame)

    return video_num


if __name__ == "__main__":

    num_workers = os.cpu_count()  # Use all available CPU cores
    print(f"Using {num_workers} workers")

    tasks = [(i, link) for i, link in enumerate(links, start=1)]

    with ProcessPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(process_video, task) for task in tasks]

        for future in as_completed(futures):
            video_num = future.result()
            print(f"Completed video {video_num}")

    print("All keyframes saved successfully.")

Using 80 workers


BrokenProcessPool: A process in the process pool was terminated abruptly while the future was running or pending.

In [6]:
import os
import cv2

# Create the main output folder
output_dir = "keyframes"
os.makedirs(output_dir, exist_ok=True)

for video_num, link in enumerate(links, start=1):

    keyframes, indices, score, z, alpha, beta = extract_keyframes(
        link,
        z_threshold=1.0
    )

    print(f"Video {video_num}")
    print("Alpha :", alpha)
    print("Beta  :", beta)
    print("Number of keyframes :", len(indices))
    print(indices)

    # Create folder for this video's keyframes
    video_folder = os.path.join(output_dir, str(video_num))
    os.makedirs(video_folder, exist_ok=True)

    # Save keyframes as 1.jpg, 2.jpg, ...
    for i, frame in enumerate(keyframes, start=1):
        save_path = os.path.join(video_folder, f"{i}.jpg")
        cv2.imwrite(save_path, frame)

print("All keyframes saved successfully.")

Video 1
Alpha : 0.5434501452365869
Beta  : 0.4565498369931744
Number of keyframes : 28
[75, 77, 79, 83, 85, 87, 91, 93, 95, 98, 100, 102, 103, 104, 106, 107, 108, 109, 110, 112, 114, 116, 118, 120, 123, 125, 133, 143]
Video 2
Alpha : 0.5321314893175018
Beta  : 0.46786848897361333
Number of keyframes : 19
[2, 6, 9, 11, 13, 15, 17, 19, 21, 23, 25, 27, 29, 31, 34, 46, 79, 121, 122]
Video 3
Alpha : 0.428020533548438
Beta  : 0.5719794457895055
Number of keyframes : 51
[2, 4, 6, 8, 52, 60, 81, 102, 110, 112, 114, 121, 123, 131, 133, 135, 144, 146, 148, 152, 158, 160, 162, 164, 169, 171, 175, 179, 181, 185, 194, 196, 200, 202, 212, 214, 219, 221, 223, 225, 227, 231, 233, 235, 237, 239, 241, 244, 246, 252, 256]
Video 4
Alpha : 0.5664252674674776
Beta  : 0.4335747103380243
Number of keyframes : 45
[137, 139, 141, 142, 143, 144, 145, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 160, 162, 164, 166, 168, 170, 172, 174, 176, 180, 182, 185, 187, 189, 191, 193, 195, 197, 201, 203, 207, 210,

KeyboardInterrupt: 

In [5]:
import os
import cv2
from concurrent.futures import ThreadPoolExecutor, as_completed

output_dir = "keyframes"
os.makedirs(output_dir, exist_ok=True)

def process_video(video_num, link):
    video_folder = os.path.join(output_dir, str(video_num))
    print(video_num)
    # Skip if folder already contains keyframes
    if os.path.isdir(video_folder) and os.listdir(video_folder):
        return f"Video {video_num} - Skipped"

    os.makedirs(video_folder, exist_ok=True)

    keyframes, indices, score, z, alpha, beta = extract_keyframes(
        link,
        z_threshold=1.0
    )

    for i, frame in enumerate(keyframes, start=1):
        cv2.imwrite(
            os.path.join(video_folder, f"{i}.jpg"),
            frame
        )

    return f"Video {video_num} - Done"


num_workers = 20
print(f"Using {num_workers} workers")

with ThreadPoolExecutor(max_workers=num_workers) as executor:
    futures = [
        executor.submit(process_video, video_num, link)
        for video_num, link in enumerate(links, start=1)
    ]

    for future in as_completed(futures):
        print(future.result())

print("All keyframes saved successfully.")

Using 20 workers
1
23

4
5
6
78
9
10
1112

13
14

15
16
1718
1920


21
22
23
24
2526

27
28
29
30
31
32
333435
36
37

3839


40
4142

4344
45
46

47
4849
50

5152

53
54
55
5657

5859
6061
6263



64
656667


68
6970

71
72
73
74
75
7677

7879

80
81
8283
84

8586

87
88
8990

91
9293

94
95
96
97
98
99
100
101102

103
104
105
106
107108

109
110111

112
113
114115
116

117118
119
120121
122

123

124125

126
127128
129
130131


132133
134

135
136
137
138139
140

141
142143
144
145146


147
148149

150
151
152
153
154155156


157158

159
160
161
162163
164165
166167168




169170

171172
173

174
175
176
177
178
179
180181
182
183

184
185
186187
188
189

190
191
192
193194
195

196197
198

199200
201
202

203204
205

206
207
208
209
210
211212213

214215


216
217
218
219
220
221222

223
224225

226
227
228
229
230
231
232
233
234235236237

238

239
240

241
242243244
245


246247248


249
250
251
252
253
254
255
256
257
258259

260261
262
263

264
265
266
267
Video 163 - Skipped
Vid

AttributeError: 'Functional' object has no attribute 'predict_function'